# 1. Carga y exploración inicial de los datos
Carga el archivo CSV con la matriz de datos urbanos y espaciales de Bogotá, mostrando una vista previa de su estructura.

In [60]:
import pandas as pd
import numpy as np
import geopandas as gpd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [61]:
print("--- 1. CARGA Y DESCRIPCIÓN EXPLORATORIA DE DATOS ---")

# Ruta del archivo unificado en el proyecto
matriz="../data/matriz_barrios_unificada.csv"
df= pd.read_csv(matriz)
display(df.head())

--- 1. CARGA Y DESCRIPCIÓN EXPLORATORIA DE DATOS ---


,localidad,barrio,v_ref_2015,v_ref_2025,v_ref_2026,cagr_15_25,num_estaciones_transmilenio,num_listings_airbnb,geometry
0,USME,EL BOSQUE,290000.0,650000.0,660000.0,0.084056,0,0.0,POLYGON ((-74.10014659299998 4.505445796000060...
1,SUBA,BILBAO,750000.0,1790000.0,2050000.0,0.090886,0,0.0,"POLYGON ((-74.113150507 4.751871279999989, -74..."
2,KENNEDY,CIUDAD KENNEDY CENTRAL,1000000.0,2900000.0,2640000.0,0.112346,0,7.0,POLYGON ((-74.15600688799998 4.616521243000022...
3,CHAPINERO,CONCEPCION NORTE,1400000.0,3580000.0,4380000.0,0.098438,1,0.0,POLYGON ((-74.06568096400002 4.656363177999992...
4,PUENTE ARANDA,COLON,1300000.0,2850000.0,3170000.0,0.081658,0,16.0,POLYGON ((-74.11963554699997 4.619638797999983...


In [62]:
def auditar_datos_inmobiliarios(df, etapa="ANTES DE LA IMPUTACIÓN"):
    """
    Función para auditar el estado de los valores de referencia y cagr.
    Muestra conteo de nulos, registros afectados y desglose específico de anomalías.
    """
    print(f"\n================ CLINICAL AUDIT: {etapa} ================")
    
    cols_eval = ['v_ref_2015', 'v_ref_2025', 'v_ref_2026', 'cagr_15_25']
    
    # 1. Conteo total de valores nulos (NaN) actuales por columna
    print("\n1. Conteo de valores nulos (NaN) por columna:")
    print(df[cols_eval].isnull().sum())
    
    # 2. Conteo de ceros (0) que representan datos faltantes disfrazados
    print("\n2. Conteo de valores en cero (0) por columna:")
    print((df[cols_eval] == 0).sum())
    
    # 3. Conteo específico por año de anomalías (NaN o 0)
    print("\n3. Desglose detallado de anomalías (NaN o 0) por año:")
    for col in cols_eval:
        n_nans = df[col].isnull().sum()
        n_zeros = (df[col] == 0).sum()
        print(f"   - {col}: {n_nans} NaNs | {n_zeros} ceros (Total anomalías: {n_nans + n_zeros})")
    
    # 4. Filtrar filas con anomalías en el rango clave del CAGR (2015 a 2025) y 2026
    condicion_critica = (
        df['v_ref_2015'].isnull() | (df['v_ref_2015'] == 0) |
        df['v_ref_2025'].isnull() | (df['v_ref_2025'] == 0) |
        df['v_ref_2026'].isnull() | (df['v_ref_2026'] == 0) |
        df['cagr_15_25'].isnull() | (df['cagr_15_25'] == 0) 
    )
    filas_criticas = df[condicion_critica]
    print(f"\n4. Total de registros afectados con anomalías entre 2015 - 2025, 2026 y cagr: {len(filas_criticas)}")
    
    # 5. Mostrar una muestra de los registros afectados si los hay
    if len(filas_criticas) > 0:
        print("\n5. Muestra de registros con anomalías críticas:")
        display(filas_criticas[['localidad', 'barrio', 'v_ref_2015', 'v_ref_2025', 'v_ref_2026', 'cagr_15_25']].head(10))
    else:
        print("\n5. ¡No se encontraron registros críticos!")
    print("================================================================")

In [63]:
auditar_datos_inmobiliarios(df, etapa="ESTADO INICIAL")


================ CLINICAL AUDIT: ESTADO INICIAL ================

1. Conteo de valores nulos (NaN) por columna:
v_ref_2015    0
v_ref_2025    0
v_ref_2026    8
cagr_15_25    0
dtype: int64

2. Conteo de valores en cero (0) por columna:
v_ref_2015    0
v_ref_2025    0
v_ref_2026    0
cagr_15_25    0
dtype: int64

3. Desglose detallado de anomalías (NaN o 0) por año:
   - v_ref_2015: 0 NaNs | 0 ceros (Total anomalías: 0)
   - v_ref_2025: 0 NaNs | 0 ceros (Total anomalías: 0)
   - v_ref_2026: 8 NaNs | 0 ceros (Total anomalías: 8)
   - cagr_15_25: 0 NaNs | 0 ceros (Total anomalías: 0)

4. Total de registros afectados con anomalías entre 2015 - 2025, 2026 y cagr: 8

5. Muestra de registros con anomalías críticas:


,localidad,barrio,v_ref_2015,v_ref_2025,v_ref_2026,cagr_15_25
84,FONTIBON,EL PANTANO,92344.5,34450.00,NaN,-0.093896
166,TEUSAQUILLO,PARQUE DISTRITAL SALITRE,80000.0,139363.69,NaN,0.057075
505,CIUDAD BOLIVAR,EL MOCHUELO III RURAL,27996.0,55498.50,NaN,0.070825
570,KENNEDY,OSORIO XII RURAL,75000.0,111000.00,NaN,0.039983
625,USME,EL MOCHUELO ALTO RURAL,27996.0,69364.00,NaN,0.094974
702,KENNEDY,OSORIO,75000.0,111000.00,NaN,0.039983
873,ENGATIVA,CAMPO EUCARISTICO,80000.0,170930.00,NaN,0.078879
986,USAQUEN,MIRADOR DEL NORTE,80000.0,138000.00,NaN,0.056036


# 2. Auditoría y separación inicial del dataset
Separa el conjunto de datos principal en datos supervisados (que tienen el valor de referencia de 2026 real) y el set de inferencia (los registros huérfanos sin 2026), conservando intactos los nombres de los barrios y las geometrías.

In [64]:
df_supervised = df[df['v_ref_2026'].notnull()].copy()
df_inference = df[df['v_ref_2026'].isnull()].copy()

print(f"Registros para entrenamiento/prueba: {len(df_supervised)}")
print(f"Registros huérfanos para predecir (Inferencia): {len(df_inference)}")

Registros para entrenamiento/prueba: 1090
Registros huérfanos para predecir (Inferencia): 8


# 3. Preparación de la matriz de características (X) y vector objetivo (y)
Prepara la matriz de entrenamiento excluyendo únicamente las variables que el modelo no procesa directamente (barrio, geometry, v_ref_2015, y el target v_ref_2026) y aplica One-Hot Encoding a la localidad.

In [65]:
# Preparar Matriz X (features) y Vector y (target) para el set supervisado
y_sup = df_supervised['v_ref_2026']

# Excluir columnas no numéricas o estructurales para el entrenamiento del modelo
cols_a_excluir = ['barrio', 'geometry', 'v_ref_2015', 'v_ref_2026']
X_sup = df_supervised.drop(columns=[col for col in cols_a_excluir if col in df_supervised.columns])

# Aplicar One-Hot Encoding a la variable categórica 'localidad'
X_sup = pd.get_dummies(X_sup, columns=['localidad'], drop_first=True, dtype=int)
display(X_sup.head())

,v_ref_2025,cagr_15_25,num_estaciones_transmilenio,num_listings_airbnb,localidad_BARRIOS UNIDOS,localidad_BOSA,localidad_CANDELARIA,localidad_CHAPINERO,localidad_CIUDAD BOLIVAR,localidad_ENGATIVA,...,localidad_LOS MARTIRES,localidad_PUENTE ARANDA,localidad_RAFAEL URIBE URIBE,localidad_SAN CRISTOBAL,localidad_SANTA FE,localidad_SUBA,localidad_TEUSAQUILLO,localidad_TUNJUELITO,localidad_USAQUEN,localidad_USME
0,650000.0,0.084056,0,0.0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
1,1790000.0,0.090886,0,0.0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,2900000.0,0.112346,0,7.0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,3580000.0,0.098438,1,0.0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
4,2850000.0,0.081658,0,16.0,0,0,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0


In [66]:
df_supervised

,localidad,barrio,v_ref_2015,v_ref_2025,v_ref_2026,cagr_15_25,num_estaciones_transmilenio,num_listings_airbnb,geometry
0,USME,EL BOSQUE,290000.0,650000.0,660000.0,0.084056,0,0.0,POLYGON ((-74.10014659299998 4.505445796000060...
1,SUBA,BILBAO,750000.0,1790000.0,2050000.0,0.090886,0,0.0,"POLYGON ((-74.113150507 4.751871279999989, -74..."
2,KENNEDY,CIUDAD KENNEDY CENTRAL,1000000.0,2900000.0,2640000.0,0.112346,0,7.0,POLYGON ((-74.15600688799998 4.616521243000022...
3,CHAPINERO,CONCEPCION NORTE,1400000.0,3580000.0,4380000.0,0.098438,1,0.0,POLYGON ((-74.06568096400002 4.656363177999992...
4,PUENTE ARANDA,COLON,1300000.0,2850000.0,3170000.0,0.081658,0,16.0,POLYGON ((-74.11963554699997 4.619638797999983...
...,...,...,...,...,...,...,...,...,...
1093,CIUDAD BOLIVAR,MEISSEN,64000.0,1198158.0,1395000.0,0.340397,0,1.0,POLYGON ((-74.13462513299999 4.560979234999991...
1094,RAFAEL URIBE URIBE,LOS MOLINOS,450000.0,2750000.0,2600000.0,0.198428,0,0.0,POLYGON ((-74.10752949900001 4.558032486000002...
1095,SUBA,EL PLAN,1400000.0,3400000.0,3798483.0,0.092786,0,38.0,POLYGON ((-74.06395278399998 4.741811476000009...
1096,CIUDAD BOLIVAR,ISMAEL PERDOMO,400000.0,1040000.0,1050000.0,0.100265,0,0.0,POLYGON ((-74.16717365199997 4.581548365999993...


# 4. División de datos en conjuntos de entrenamiento y prueba
Divide la matriz supervisada en conjuntos de entrenamiento (80%) y prueba (20%) para evaluar de manera objetiva el rendimiento del modelo.

In [67]:
X_train, X_test, y_train, y_test = train_test_split(X_sup, y_sup, test_size=0.2, random_state=42)
print(f"Tamaño X_train: {X_train.shape}, X_test: {X_test.shape}")

Tamaño X_train: (872, 22), X_test: (218, 22)


# 5. Entrenamiento y evaluación del modelo Random Forest
Inicializa y entrena el algoritmo Random Forest Regressor con hiperparámetros optimizados para estabilidad espacial, calculando las métricas de desempeño ($R^2$, RMSE, MAE y MSE).

In [68]:
# Inicializar y entrenar el Random Forest
rf_model = RandomForestRegressor(n_estimators=300,          # Mayor cantidad de árboles para estabilidad
                                 max_depth=15,              # Límite de profundidad para prevenir overfitting
                                 min_samples_split=5,       # Mínimo de muestras para dividir un nodo
                                 min_samples_leaf=2,        # Mínimo de muestras por hoja
                                 max_features='sqrt',       # Submuestreo de features por división (robusto espacialmente)
                                 random_state=42,
                                 n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predecir sobre el set de prueba
y_pred_test = rf_model.predict(X_test)

# Calcular métricas de desempeño
mse_rf = mean_squared_error(y_test, y_pred_test)
rmse_rf = np.sqrt(mse_rf)
mae_rf = mean_absolute_error(y_test, y_pred_test)
r2_rf = r2_score(y_test, y_pred_test)

print("--- RESULTADOS RANDOM FOREST OPTIMIZADO ---")
print(f"MSE:  {mse_rf:,.2f}")
print(f"RMSE: {rmse_rf:,.2f}")
print(f"MAE:  {mae_rf:,.2f}")
print(f"R2:   {r2_rf:.4f}")

--- RESULTADOS RANDOM FOREST OPTIMIZADO ---
MSE:  164,026,108,871.47
RMSE: 405,001.37
MAE:  285,014.11
R2:   0.9415


# 6. Inferencia, consolidación y formato final para Dash y la API
Realiza la predicción sobre los registros huérfanos y supervisados, consolida el dataset maestro conservando los barrios y geometrías, calcula la hipervalorización y ajusta los formatos numéricos.

In [69]:
# 1. Recuperar la geometría original que habíamos descartado al inicio en 'df_supervised' y 'df_inference'
X_inf = df_inference.drop(columns=['v_ref_2026'])
X_inf = pd.get_dummies(X_inf, columns=['localidad'], drop_first=True, dtype=int)
X_inf = X_inf.reindex(columns=X_train.columns, fill_value=0)

# Predicción para los 8 registros huérfanos
df_inference['v_ref_2026_predicho'] = rf_model.predict(X_inf)

# 2. Generar predicción para todo el set supervisado
X_sup_completo = df_supervised.drop(columns=['v_ref_2026'])
X_sup_completo = pd.get_dummies(X_sup_completo, columns=['localidad'], drop_first=True, dtype=int)
X_sup_completo = X_sup_completo.reindex(columns=X_train.columns, fill_value=0)

# 1. Predicción para los 8 registros huérfanos
X_inf = df_inference.drop(columns=[col for col in cols_a_excluir if col in df_inference.columns])
X_inf = pd.get_dummies(X_inf, columns=['localidad'], drop_first=True, dtype=int)
X_inf = X_inf.reindex(columns=X_train.columns, fill_value=0)
df_inference['v_ref_2026_predicho'] = rf_model.predict(X_inf)

# 2. Generar predicción para todo el set supervisado
X_sup_completo = df_supervised.drop(columns=[col for col in cols_a_excluir if col in df_supervised.columns])
X_sup_completo = pd.get_dummies(X_sup_completo, columns=['localidad'], drop_first=True, dtype=int)
X_sup_completo = X_sup_completo.reindex(columns=X_train.columns, fill_value=0)
df_supervised['v_ref_2026_predicho'] = rf_model.predict(X_sup_completo)

# 3. Unificar todo en un solo DataFrame maestro consolidado (con barrios y geometría intactos)
df_final_maestro = pd.concat([df_supervised, df_inference], ignore_index=True)

# 4. Calcular métricas clave para el Dashboard en Dash
# Porcentaje de hipervalorización proyectada usando el valor predicho
df_final_maestro['hipervalorizacion_pct'] = ((df_final_maestro['v_ref_2026_predicho'] - df_final_maestro['v_ref_2025']) / df_final_maestro['v_ref_2025']) * 100

# Clasificación de los datos para filtros en Dash
df_final_maestro['tipo_registro'] = df_final_maestro['v_ref_2026'].apply(lambda x: 'Histórico Oficial' if pd.notnull(x) else 'Imputado / Predicho (Huérfano)')

# 5. Ajustar el precio predicho a número normal (redondeado a 0 decimales ya que son pesos en COP)
df_final_maestro['v_ref_2026_predicho'] = df_final_maestro['v_ref_2026_predicho'].round(0)

# 6. Ajustar la hipervalorización en porcentaje estrictamente a 2 decimales
df_final_maestro['hipervalorizacion_pct'] = df_final_maestro['hipervalorizacion_pct'].round(2)

# 7. Desactivar la notación científica globalmente en Pandas para que se vea bien en el notebook
pd.set_option('display.float_format', lambda x: '%.2f' % x)

display(df_final_maestro.head())

,localidad,barrio,v_ref_2015,v_ref_2025,v_ref_2026,cagr_15_25,num_estaciones_transmilenio,num_listings_airbnb,geometry,v_ref_2026_predicho,hipervalorizacion_pct,tipo_registro
0,USME,EL BOSQUE,290000.00,650000.00,660000.00,0.08,0,0.00,POLYGON ((-74.10014659299998 4.505445796000060...,630976.00,-2.93,Histórico Oficial
1,SUBA,BILBAO,750000.00,1790000.00,2050000.00,0.09,0,0.00,"POLYGON ((-74.113150507 4.751871279999989, -74...",2182465.00,21.93,Histórico Oficial
2,KENNEDY,CIUDAD KENNEDY CENTRAL,1000000.00,2900000.00,2640000.00,0.11,0,7.00,POLYGON ((-74.15600688799998 4.616521243000022...,3209363.00,10.67,Histórico Oficial
3,CHAPINERO,CONCEPCION NORTE,1400000.00,3580000.00,4380000.00,0.10,1,0.00,POLYGON ((-74.06568096400002 4.656363177999992...,4318431.00,20.63,Histórico Oficial
4,PUENTE ARANDA,COLON,1300000.00,2850000.00,3170000.00,0.08,0,16.00,POLYGON ((-74.11963554699997 4.619638797999983...,3408244.00,19.59,Histórico Oficial


# 7. Exportación de archivos para Dash y consumo por API
Exporta el resultado final en formato GeoPackage (.gpkg) con sus respectivas geometrías de polígonos para visualizar en Dash, y en formato Parquet para la API.

In [70]:
# 8. Exportación de archivos finales adaptados para Dash y API
# A. Convertir el DataFrame maestro en un GeoDataFrame real usando los polígonos
#gdf_export = gpd.GeoDataFrame(df_final_maestro, geometry=gpd.GeoSeries.from_wkt(df_final_maestro['geometry']), crs="EPSG:4326")

# B. Exportar en formato .gpkg (GeoPackage) - Recomendado para Dash y mapas espaciales
#gdf_export.to_file("output_hipervalorizacion_bogota_2026.gpkg", driver="GPKG")
#print("\n¡Archivo geoespacial en .gpkg exportado con éxito para tu dashboard en Dash!")

# C. Exportar versión tabular sin geometría (o ligera) por si tu API la requiere para consultas rápidas
#df_final_maestro.drop(columns=['geometry']).to_parquet("datos_api_bogota.parquet", index=False)
#print("¡Archivo tabular para la API exportado con éxito: 'datos_api_bogota.parquet'!")